# Semana 6: inferência nos alvos, 15 execuções, códigos de regime sorteados

Seção 5 e seção 10 de `prereg/preregistro.md` (commit `26cc8b7`). Roda as 15 execuções do treino confirmatório (R1, R2 e R3, sementes 1 a 5, pesos em `Mestrado/resultados/semana5`) em 32 bits, em chips e janelas inteiros de 512 × 512, na validação, no teste da fonte, na Bolívia e no T2 (as 441 janelas da sensibilidade do conjunto completo, que contêm as do núcleo, mais as 120 janelas dos mapas da sensibilidade de nuvem 0,50). Para cada pixel pontuado grava a margem de logit m = z_água − z_terra.

**Os alvos são lidos aqui.** O script de análise já está congelado (`analysis/README.md`, commit `f9de2f5` e seguintes) e as janelas de nuvem 0,50 já estão no repositório.

**Cegamento.** As margens são gravadas sob três códigos de quatro caracteres sorteados neste notebook. A chave (código para R1, R2 ou R3) vai para `Mestrado/resultados/semana6_chave/chave_codigos.json`, uma pasta à parte. O notebook não mostra a chave, não imprime nenhuma métrica dos alvos e esconde o que a construção de cada modelo imprime. **Não abra a pasta `semana6_chave`**: a chave só é lida pelo script de análise, na rodada única. O cegamento é procedimental, e isso é dito.

| O que roda | Detalhe |
|---|---|
| Entrada | 13 bandas do Sen1Floods11 (val, teste, Bolívia) com os índices 1, 2, 3, 8, 11 e 12; janelas de 6 bandas do T2 já cortadas; escala 1e-4 e normalização do próprio módulo de dados do TerraTorch, como no treino |
| Rótulo | 1 água, 0 terra, −1 ignorado; só os pixels 0 e 1 são pontuados, em ordem de linhas |
| Borda | `borda` marca o pixel de água ou terra que tem, a até 2 pixels (janela de 5 × 5), um pixel da outra classe; pixel −1 não forma borda. No T2 a borda é calculada na cena inteira e depois recortada. A definição não estava no pré-registro e vale só para a sensibilidade sem fronteira |
| Conferência | A inferência da validação é refeita com os pesos salvos e comparada com a que o treino gravou: a diferença máxima tem de ficar abaixo de 0,05 |
| Saída | `resultados/semana6/margens`: `labels_<conjunto>.npz`, `run_<código>_s<semente>_<conjunto>.npz`, `manifesto_sha256.csv`, `info_inferencia.json` |

**Fora deste notebook:** o Rio Grande do Sul (RQ4), que precisa dos tiles e de um script próprio, e o próprio script de análise.

Abra o Colab logada na conta do mestrado, com GPU T4, e rode na ordem. Se a sessão cair ou a RAM passar de 62%, a célula de inferência para; reinicie a sessão e rode de novo as células 2, 3, 4, 5, 6, 7, 8 e 9, **sem pular a 6**: o que está gravado é pulado. A célula 1 só precisa rodar na primeira sessão ou se o Colab pedir. Guarde a saída das células 6, 9 e 11. Se alguma célula der erro, pare e guarde o erro.

Etapas: 1 instalação; 2 Drive e treinos encontrados; 3 backbone conferido pelo hash; 4 listas e imagens do Sen1Floods11; 5 funções de leitura; 6 leitura dos quatro conjuntos e gravação dos rótulos; 7 definições do modelo; 8 sorteio dos códigos; 9 inferência; 10 aumento em tempo de teste (opcional); 11 conferência final.

In [ ]:
!pip install -q "terratorch[peft]==1.2.11" "torchgeo==0.7.1" "peft==0.21.2"
!pip uninstall -y -q torchao

from importlib.metadata import version
for pacote in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy'):
    print(f'{pacote:11s} {version(pacote)}')
assert version('terratorch') == '1.2.11', 'Versão do TerraTorch diferente da fixada.'

import sys
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != version('numpy'):
    print(f"\nREINICIE A SESSÃO: o NumPy carregado é o {sys.modules['numpy'].__version__} e o instalado é o {version('numpy')}.")
    print('Menu Ambiente de execução > Reiniciar sessão (não precisa excluir o ambiente) e rode a partir da célula 2.')
else:
    print('\nNumPy carregado e instalado coincidem. Pode seguir para a célula 2.')

In [ ]:
import os, re, json, time, hashlib, datetime, shutil, secrets, glob, gc, csv
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

S5 = f'{RAIZ}/resultados/semana5'
S2R = f'{RAIZ}/resultados/semana2'
SAIDA = f'{RAIZ}/resultados/semana6/margens'
CHAVE_DIR = f'{RAIZ}/resultados/semana6_chave'
JANELAS = f'{RAIZ}/dados/worldfloods_v2/janelas'
for d in (SAIDA, CHAVE_DIR):
    os.makedirs(d, exist_ok=True)
for resto in glob.glob(f'{SAIDA}/parcial_*'):
    os.remove(resto)

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

CHAVE_RX = re.compile(r'([A-Za-z-]+_\d+)')
def chave_de(nome):
    achado = CHAVE_RX.search(os.path.basename(str(nome)))
    assert achado, f'Nome de chip sem chave reconhecível: {nome!r}'
    return achado.group(1)

REGIMES = ('R1', 'R2', 'R3')
SEMENTES = (1, 2, 3, 4, 5)
PESOS = {}
for regime in REGIMES:
    for semente in SEMENTES:
        boas = []
        for tentativa in (1, 2):
            registro = f'{S5}/exec_{regime}_s{semente}_t{tentativa}.json'
            if os.path.exists(registro) and json.load(open(registro)).get('situacao') == 'ok':
                boas.append(registro)
        pesos, margens = f'{S5}/pesos_{regime}_s{semente}.pt', f'{S5}/margens_{regime}_s{semente}.npz'
        assert len(boas) == 1 and os.path.exists(pesos) and os.path.exists(margens), (
            f'Faltam arquivos do treino em {S5}. Confira a pasta resultados/semana5.')
        PESOS[(regime, semente)] = dict(pesos=pesos, margens_val=margens)
print(f'{len(PESOS)} execuções de treino encontradas.')

livre = shutil.disk_usage(MEUDRIVE)[2]
print(f'Espaço livre no Drive: {livre / 1e9:.0f} GB. As margens ocupam cerca de 10 GB (cerca de 20 GB com o TTA).')
assert livre > 12e9, 'Espaço livre no Drive insuficiente para as margens (mínimo de 12 GB).'
print(f'Saída em {SAIDA}')

In [ ]:
from huggingface_hub import hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
ARQUIVO = 'Prithvi_EO_V2_300M_TL.pt'
REVISAO = '63adbd39c271da4c42f447e69b1a7c91a338cdc9'
SHA_BACKBONE = '3629cedfbb350faafcb0dac902ae0d3c927e25ce8d9e0024aa1276ec66956ddb'

no_drive = f'{RAIZ}/modelo_fixado/backbone/{ARQUIVO}'
if not os.path.exists(no_drive):
    print('Backbone ausente no Drive; baixando na revisão fixada.')
    hf_hub_download(REPO, ARQUIVO, revision=REVISAO, local_dir=os.path.dirname(no_drive))
BACKBONE_PT = f'/content/{ARQUIVO}'
if not (os.path.exists(BACKBONE_PT) and sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE):
    shutil.copy(no_drive, BACKBONE_PT)
assert sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE, 'sha256 do backbone diferente do registrado em modelo.md.'
print(f'Backbone conferido ({SHA_BACKBONE[:8]}…), cópia local em {BACKBONE_PT}')

## Sen1Floods11: val, teste e Bolívia

As três listas são conferidas contra os hashes de `splits/README.md`. Os 194 chips e rótulos vêm do bucket público; o hash de cada arquivo baixado fica em `resultados/semana6/entradas_sen1floods11_sha256.csv`.

In [ ]:
import urllib.request, concurrent.futures as cf

BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/'
DADOS = '/content/sen1floods11'
S2 = f'{DADOS}/v1.1/data/flood_events/HandLabeled/S2Hand'
LB = f'{DADOS}/v1.1/data/flood_events/HandLabeled/LabelHand'
SP = f'{DADOS}/v1.1/splits/flood_handlabeled'
SPLITS_DRIVE = f'{RAIZ}/dados/splits'
for d in (S2, LB, SP, SPLITS_DRIVE):
    os.makedirs(d, exist_ok=True)

LISTAS = {'val': ('flood_valid_data.csv', 89, '14a372859fb18b1f7f9791810b7a233c2cd53aaab84f74721f9ac3476ba50504'),
          'teste': ('flood_test_data.csv', 90, 'cfcf04cc36880d0b4156681c9c2efaeb26d62e7ecb6987002e437436cdf98627'),
          'bolivia': ('flood_bolivia_data.csv', 15, 'd3c9e2d98e9f6facf19c851d689567e34523449a59d1b3eb0543c516df4094c8')}

def chaves_csv(nome):
    destino = os.path.join(SPLITS_DRIVE, nome)
    if not os.path.exists(destino):
        urllib.request.urlretrieve(BUCKET + 'splits/flood_handlabeled/' + nome, destino)
    chaves = []
    for linha in open(destino).read().strip().splitlines():
        primeiro = os.path.basename(linha.split(',')[0].strip())
        chave = re.sub(r'_(S1Hand|S2Hand|LabelHand)(\.tif)?$', '', primeiro)
        assert chave and chave != primeiro, f'Linha inesperada em {nome}: {linha!r}'
        chaves.append(chave)
    return chaves

CHAVES_S1F11 = {}
for conj, (arquivo, n_esperado, sha_lista) in LISTAS.items():
    chaves = chaves_csv(arquivo)
    lista = '\n'.join(sorted(f'{c}_LabelHand.tif' for c in chaves)) + '\n'
    assert len(chaves) == n_esperado and len(set(chaves)) == n_esperado, f'{conj}: {len(chaves)} chips, esperado {n_esperado}.'
    assert hashlib.sha256(lista.encode()).hexdigest() == sha_lista, f'Lista de {conj} diferente da registrada em splits/README.md.'
    CHAVES_S1F11[conj] = chaves
todas = [c for lista in CHAVES_S1F11.values() for c in lista]
assert len(set(todas)) == len(todas), 'Há chip em mais de uma lista.'

def baixar(url_e_destino):
    url, destino = url_e_destino
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return 0
    for tentativa in range(4):
        try:
            urllib.request.urlretrieve(url, destino + '.parcial')
            os.replace(destino + '.parcial', destino)
            return 1
        except Exception as erro:
            ultimo = erro
            time.sleep(2 * (tentativa + 1))
    raise RuntimeError(f'Falhou: {url} ({ultimo})')

trabalhos = []
for chave in todas:
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/S2Hand/{chave}_S2Hand.tif', f'{S2}/{chave}_S2Hand.tif'))
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/LabelHand/{chave}_LabelHand.tif', f'{LB}/{chave}_LabelHand.tif'))
t0 = time.perf_counter()
with cf.ThreadPoolExecutor(16) as pool:
    novos = sum(pool.map(baixar, trabalhos))
print(f'{len(trabalhos)} arquivos conferidos, {novos} baixados agora, em {time.perf_counter() - t0:.0f} s.')

with open(os.path.join(os.path.dirname(SAIDA), 'entradas_sen1floods11_sha256.csv'), 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['arquivo', 'bytes', 'sha256'])
    for _, destino in trabalhos:
        w.writerow([os.path.basename(destino), os.path.getsize(destino), sha256_arquivo(destino)])
print('Lista de hashes das imagens e rótulos gravada em resultados/semana6.')

## Leitura dos conjuntos

A célula 5 só define funções. A célula 6 lê os quatro conjuntos (a primeira leitura das 561 janelas do Drive leva alguns minutos), confere as contagens de cada janela contra `t2_janelas.csv` e `t2_janelas_nuvem050.csv`, grava os `labels_<conjunto>.npz` e confere que os rótulos da validação são os mesmos que o treino usou.

In [ ]:
import rasterio
from scipy.ndimage import maximum_filter

TAM = 512
IDX_BANDAS = [1, 2, 3, 8, 11, 12]          # B2, B3, B4, B8A, B11, B12 nos 13 canais do Sen1Floods11
ESCALA = np.float32(0.0001)
RAIO_BORDA = 2

def borda_de(y):
    """Verdadeiro nos pixels de água ou terra a até 2 pixels (janela 5 x 5) de um pixel da outra classe.
    Pixels com rótulo -1 não formam borda."""
    agua = (y == 1).astype(np.uint8)
    terra = (y == 0).astype(np.uint8)
    lado = 2 * RAIO_BORDA + 1
    perto_agua = maximum_filter(agua, size=lado, mode='constant', cval=0).astype(bool)
    perto_terra = maximum_filter(terra, size=lado, mode='constant', cval=0).astype(bool)
    return ((agua == 1) & perto_terra) | ((terra == 1) & perto_agua)

PASTA_CACHE = '/content/cache_imagens'

def alocar(nome, forma, tipo):
    """Matriz de imagens no disco local (memmap), para não ocupar a RAM da sessão."""
    os.makedirs(PASTA_CACHE, exist_ok=True)
    return np.lib.format.open_memmap(f'{PASTA_CACHE}/{nome}.npy', mode='w+', dtype=tipo, shape=forma)

def preencher(x, valor):
    x = np.ma.filled(x.astype(np.float32), np.nan)
    return np.where(np.isnan(x), np.float32(valor), x).astype(np.float32)

def ler_chip(chave, pasta_s2, pasta_lb):
    with rasterio.open(f'{pasta_s2}/{chave}_S2Hand.tif') as s:
        assert s.count == 13 and (s.height, s.width) == (TAM, TAM), f'{chave}: imagem {s.count} x {s.height} x {s.width}'
        x = preencher(s.read(masked=True), 0.0)[IDX_BANDAS]
    with rasterio.open(f'{pasta_lb}/{chave}_LabelHand.tif') as s:
        assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{chave}: rótulo {s.count} x {s.height} x {s.width}'
        y = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
    assert set(np.unique(y).tolist()) <= {-1, 0, 1}, f'{chave}: valores de rótulo {np.unique(y)}'
    return x, y

def montar_s1f11(chaves, pasta_s2, pasta_lb, nome='s1f11'):
    n = len(chaves)
    bruto = alocar(nome, (n, len(IDX_BANDAS), TAM, TAM), np.float32)
    ys = np.empty((n, TAM, TAM), np.int16)
    bordas = np.empty((n, TAM, TAM), bool)
    for i, c in enumerate(chaves):
        bruto[i], ys[i] = ler_chip(c, pasta_s2, pasta_lb)
        bordas[i] = borda_de(ys[i])
    bruto.flush()
    return dict(chaves=list(chaves), bruto=bruto, ys=ys, bordas=bordas)

def montar_t2(linhas, pasta_janelas, pasta_wf2):
    """linhas: linhas das tabelas de janelas que entram na sensibilidade, em qualquer ordem."""
    linhas = sorted(linhas, key=lambda r: r['window_id'])
    n = len(linhas)
    bruto = alocar('t2', (n, len(IDX_BANDAS), TAM, TAM), np.int16)
    ys = np.empty((n, TAM, TAM), np.int16)
    bordas = np.empty((n, TAM, TAM), bool)
    posicao = {r['window_id']: i for i, r in enumerate(linhas)}
    assert len(posicao) == n, 'Janelas repetidas.'
    for mapa in sorted({r['map_id'] for r in linhas}):
        with rasterio.open(f'{pasta_wf2}/test/gt/{mapa}.tif') as g:
            gt = g.read()
        cena = np.full(gt.shape[1:], -1, np.int16)
        cena[gt[1] == 1] = 0
        cena[gt[1] == 2] = 1
        cena[gt[0] != 1] = -1
        borda_cena = borda_de(cena)
        for r in (r for r in linhas if r['map_id'] == mapa):
            i, jid = posicao[r['window_id']], r['window_id']
            lin, col = int(r['row_off']), int(r['col_off'])
            with rasterio.open(f'{pasta_janelas}/S2/{jid}.tif') as s:
                assert s.count == 6 and (s.height, s.width) == (TAM, TAM) and s.dtypes[0] == 'int16', f'{jid}: imagem'
                bruto[i] = s.read()
            with rasterio.open(f'{pasta_janelas}/label/{jid}.tif') as s:
                assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{jid}: rótulo'
                y = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
            ys[i] = y
            recorte = cena[lin:lin + TAM, col:col + TAM]
            pont = y >= 0
            assert np.array_equal(y[pont], recorte[pont]), f'{jid}: rótulo da janela diferente do recorte da cena'
            bordas[i] = borda_cena[lin:lin + TAM, col:col + TAM]
            n_pont, n_agua = int(pont.sum()), int((y == 1).sum())
            assert (n_pont, n_agua, n_pont - n_agua) == (int(r['scored_pixels']), int(r['water_pixels']), int(r['land_pixels'])), (
                f'{jid}: contagens diferentes das da tabela de janelas')
    bruto.flush()
    return dict(chaves=[r['window_id'] for r in linhas], bruto=bruto, ys=ys, bordas=bordas)

def rotulos_do_conjunto(c, com_borda):
    pont = c['ys'] >= 0
    saida = dict(chaves=np.array(c['chaves']), tamanhos=pont.reshape(len(c['chaves']), -1).sum(1).astype(np.int64),
                 rotulo=c['ys'][pont].astype(np.uint8))
    if com_borda:
        saida['borda'] = c['bordas'][pont]
    return saida

def gravar(caminho, **campos):
    tmp = os.path.join(os.path.dirname(caminho), 'parcial_' + os.path.basename(caminho))
    np.savez(tmp, **campos)
    os.replace(tmp, caminho)

In [ ]:
def ler_tabela(nome, sha_esperado):
    caminho = f'{S2R}/{nome}'
    assert sha256_arquivo(caminho) == sha_esperado, f'{nome} do Drive não é o registrado no repositório.'
    with open(caminho, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

TAB_JANELAS = ler_tabela('t2_janelas.csv', '3c21970835f2d43cd436be9616c451ebbff5fcea138b7401d3867f0901643b2c')
TAB_NUVEM = ler_tabela('t2_janelas_nuvem050.csv', '3b46e703738042b235b68a12600cfc0c7bf917b3cf5dec7efd8b924a02e132a6')
entram = [r for r in TAB_JANELAS if r['kept_sensitivity'] == 'True']
entram_nuvem = [r for r in TAB_NUVEM if r['kept_sensitivity'] == 'True']
assert (len(entram), len(entram_nuvem)) == (441, 120), (len(entram), len(entram_nuvem))
assert not {r['window_id'] for r in entram} & {r['window_id'] for r in entram_nuvem}

WF2 = '/content/wf2' if os.path.isdir('/content/wf2/test/gt') else f'{RAIZ}/dados/worldfloods_v2'
t0 = time.perf_counter()
CONJ = {}
for conj in ('val', 'teste', 'bolivia'):
    CONJ[conj] = montar_s1f11(CHAVES_S1F11[conj], S2, LB, nome=conj)
    print(f'{conj}: {len(CONJ[conj]["chaves"])} chips lidos ({time.perf_counter() - t0:.0f} s)')
CONJ['t2'] = montar_t2(entram + entram_nuvem, JANELAS, WF2)
print(f't2: {len(CONJ["t2"]["chaves"])} janelas lidas ({time.perf_counter() - t0:.0f} s)')

ROTULOS = {}
for conj, c in CONJ.items():
    r = rotulos_do_conjunto(c, com_borda=(conj != 'val'))
    ROTULOS[conj] = r
    caminho = f'{SAIDA}/labels_{conj}.npz'
    if os.path.exists(caminho):
        antigo = np.load(caminho, allow_pickle=False)
        assert all(np.array_equal(antigo[k], r[k]) for k in r) and set(antigo.files) == set(r), (
            f'labels_{conj}.npz do Drive é diferente do recalculado. Guarde esta saída.')
    else:
        gravar(caminho, **r)
    print(f'{conj}: {len(r["chaves"])} unidades, {int((r["tamanhos"] > 0).sum())} com pixel pontuado, '
          f'{int(r["tamanhos"].sum()):,} pixels pontuados')
    if 'borda' in r:
        print(f'   pixels de borda: {100 * r["borda"].mean():.1f}%')
    c.pop('bordas')

assert int(ROTULOS['val']['tamanhos'].sum()) == 20294725, 'Os pixels pontuados da val diferem dos 20.294.725 do treino.'
assert (ROTULOS['bolivia']['tamanhos'] > 0).all() and int((ROTULOS['teste']['tamanhos'] > 0).sum()) == 89
assert int((ROTULOS['val']['tamanhos'] > 0).sum()) == 86

# os rótulos lidos aqui são os mesmos que o treino usou na val
orig = np.load(PESOS[('R1', 1)]['margens_val'], allow_pickle=False)
rot_o, tam_o, ch_o = orig['rotulo'], orig['tamanhos'], [str(c) for c in orig['chaves']]
off_o = np.concatenate([[0], np.cumsum(tam_o)])
pos_o = {c: i for i, c in enumerate(ch_o)}
off_n = np.concatenate([[0], np.cumsum(ROTULOS['val']['tamanhos'])])
for i, c in enumerate(ROTULOS['val']['chaves']):
    j = pos_o[str(c)]
    assert np.array_equal(rot_o[off_o[j]:off_o[j + 1]], ROTULOS['val']['rotulo'][off_n[i]:off_n[i + 1]]), c
print('Rótulos da val iguais aos do treino.')

## Modelo e inferência

A célula 7 monta cada regime com a mesma configuração do treino, carrega os pesos salvos (conferindo que os tensores batem um a um; no R1 o codificador vem do backbone fixado) e define a inferência em 32 bits.

In [ ]:
import warnings, contextlib, io, logging, gc
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from importlib.metadata import version
from scipy.special import log_expit, logsumexp
try:
    from terratorch.tasks import SemanticSegmentationTask
    from terratorch.datamodules import Sen1Floods11NonGeoDataModule
except ModuleNotFoundError as erro:
    raise RuntimeError('TerraTorch não está instalado nesta sessão. Rode a célula 1 e confira que ela imprime terratorch 1.2.11.') from erro

warnings.filterwarnings('ignore', category=UserWarning)
assert torch.cuda.is_available(), 'Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4.'
DISPOSITIVO = 'cuda'
BANDAS = ['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2']
ESPERADO_M = {'R1': 15.1, 'R2': 20.6, 'R3': 319.0}
TOLERANCIA_M = 0.1
LR = {'R1': 5e-4, 'R2': 5e-4, 'R3': 5e-5}
EPOCAS = 50
LOTE_INF = 8
LIMITE_DIF_VAL = 0.05

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def datamodule():
    para_tensor = lambda: ToTensorV2(transpose_mask=False)
    return Sen1Floods11NonGeoDataModule(
        data_root=DADOS, batch_size=LOTE_INF, num_workers=0, bands=BANDAS,
        train_transform=[para_tensor()], val_transform=[para_tensor()], test_transform=[para_tensor()],
        drop_last=False, constant_scale=0.0001, no_data_replace=0.0, no_label_replace=-1, use_metadata=False)

LORA = dict(method='LORA', replace_qkv='qkv',
            peft_config_kwargs=dict(target_modules=['qkv.q_linear', 'qkv.v_linear', 'mlp.fc1', 'mlp.fc2'],
                                    lora_alpha=16, r=16))

def tarefa(regime):
    args = dict(
        backbone='prithvi_eo_v2_300_tl', backbone_pretrained=True, backbone_ckpt_path=BACKBONE_PT,
        backbone_bands=BANDAS,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}],
        decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
        head_dropout=0.1, num_classes=2, rescale=True)
    if regime == 'R2':
        args['peft_config'] = LORA
    t = SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory', model_args=args, loss='ce', ignore_index=-1,
        lr=LR[regime], optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
        scheduler='CosineAnnealingLR', scheduler_hparams={'T_max': EPOCAS, 'eta_min': 0},
        freeze_backbone=(regime == 'R1'), plot_on_val=0)
    t.configure_models()
    treinaveis = sum(p.numel() for p in t.parameters() if p.requires_grad)
    if regime == 'R2':
        n_lora = sum(1 for n, _ in t.model.encoder.named_parameters() if 'lora_A' in n)
        assert n_lora == 96, 'Número de módulos LoRA diferente do esperado.'
    assert abs(treinaveis / 1e6 - ESPERADO_M[regime]) <= TOLERANCIA_M, 'Parâmetros treináveis fora do esperado.'
    return t

DM = datamodule()

@contextlib.contextmanager
def em_silencio():
    """Esconde o que a construção do modelo imprime, que diferiria entre os regimes."""
    logging.disable(logging.CRITICAL)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)

def carregar_modelo(regime, semente):
    with em_silencio():
        t = tarefa(regime)
        t.model.to(DISPOSITIVO)
        gc.collect()
        pesos = torch.load(PESOS[(regime, semente)]['pesos'], map_location=DISPOSITIVO)
        do_modelo = set(t.model.state_dict().keys())
        esperado = do_modelo if regime != 'R1' else {k for k in do_modelo if not k.startswith('encoder.')}
        assert set(pesos.keys()) == esperado, 'Os tensores salvos não coincidem com os do modelo.'
        resultado = t.model.load_state_dict(pesos, strict=False)
        assert not resultado.unexpected_keys, 'Tensores salvos sem lugar no modelo.'
        assert set(resultado.missing_keys) == (do_modelo - esperado), 'Tensores do modelo sem peso.'
        modelo = t.model.to(DISPOSITIVO).float().eval()
        for p in modelo.parameters():
            p.requires_grad_(False)
        del t, pesos
    gc.collect()
    return modelo

TRANSFORMACOES = [(k, espelho) for espelho in (False, True) for k in range(4)]

def aplicar(x, k, espelho):
    if espelho:
        x = torch.flip(x, dims=(-1,))
    return torch.rot90(x, k, dims=(-2, -1))

def desfazer(y, k, espelho):
    y = torch.rot90(y, -k, dims=(-2, -1))
    if espelho:
        y = torch.flip(y, dims=(-1,))
    return y

def margem_media_probabilidade(pilha):
    """Logit da probabilidade média de água de K margens, pilha (K, N), em float64 (igual a nucleo.mean_probability_margin)."""
    pilha = np.asarray(pilha, dtype=np.float64)
    k = pilha.shape[0]
    lp = logsumexp(log_expit(pilha), axis=0) - np.log(k)
    lq = logsumexp(log_expit(-pilha), axis=0) - np.log(k)
    return lp - lq

def inferir(modelo, conj, tta=False):
    """Margem de logit (água menos terra) de cada pixel pontuado, em 32 bits, chips inteiros.
    Com tta=True, logit da probabilidade média sobre as 8 transformações do grupo diedral."""
    bruto, ys = conj['bruto'], conj['ys']
    margens = []
    with torch.no_grad(), torch.autocast(device_type=DISPOSITIVO, enabled=False):
        for i in range(0, len(ys), LOTE_INF):
            x = torch.from_numpy(bruto[i:i + LOTE_INF].astype(np.float32) * ESCALA).to(DISPOSITIVO)
            y = torch.from_numpy(ys[i:i + LOTE_INF].astype(np.int64)).to(DISPOSITIVO)
            x = DM.aug({'image': x, 'mask': y})['image']
            assert x.dtype == torch.float32 and x.ndim == 4 and x.shape[1] == len(BANDAS), tuple(x.shape)
            pilhas = []
            for k, espelho in (TRANSFORMACOES if tta else [(0, False)]):
                z = modelo(aplicar(x, k, espelho)).output.float()
                assert tuple(z.shape[-2:]) == tuple(x.shape[-2:]), f'Saída {tuple(z.shape)}'
                pilhas.append(desfazer(z[:, 1] - z[:, 0], k, espelho).cpu().numpy())
            for j in range(x.shape[0]):
                pont = ys[i + j] >= 0
                if tta:
                    margens.append(margem_media_probabilidade(np.stack([p[j][pont] for p in pilhas])).astype(np.float32))
                else:
                    margens.append(pilhas[0][j][pont].astype(np.float32))
    return np.concatenate(margens)

def comparar_val(regime, semente, chaves, tamanhos, margem):
    z = np.load(PESOS[(regime, semente)]['margens_val'], allow_pickle=False)
    ch_o, tam_o, marg_o = [str(c) for c in z['chaves']], z['tamanhos'], z['margem']
    pos = {c: i for i, c in enumerate(ch_o)}
    off_o = np.concatenate([[0], np.cumsum(tam_o)])
    off_n = np.concatenate([[0], np.cumsum(tamanhos)])
    dif_max, trocas = 0.0, 0
    for i, c in enumerate(chaves):
        j = pos[str(c)]
        assert int(tam_o[j]) == int(tamanhos[i])
        a, b = marg_o[off_o[j]:off_o[j + 1]], margem[off_n[i]:off_n[i + 1]]
        if a.size:
            dif_max = max(dif_max, float(np.max(np.abs(a - b))))
            trocas += int(np.sum((a > 0) != (b > 0)))
    return dif_max, trocas

print('Definições carregadas. Nada foi inferido ainda.')

## Códigos de regime

Sorteia três códigos e a ordem em que as 15 execuções rodam. Se os códigos já existem de uma sessão anterior, reaproveita sem mostrar a chave.

In [ ]:
CHAVE_ARQ = f'{CHAVE_DIR}/chave_codigos.json'
if os.path.exists(CHAVE_ARQ):
    CHAVE = json.load(open(CHAVE_ARQ))
    assert sorted(CHAVE.values()) == list(REGIMES) and len(CHAVE) == 3, 'A chave dos códigos está incompleta.'
    print('Códigos de regime já sorteados em uma sessão anterior. A chave não é lida nem mostrada.')
else:
    sorteio = secrets.SystemRandom()
    alfabeto = 'abcdefghjkmnpqrstuvwxyz23456789'
    codigos = set()
    while len(codigos) < 3:
        codigos.add(''.join(sorteio.choice(alfabeto) for _ in range(4)))
    regimes = list(REGIMES)
    sorteio.shuffle(regimes)
    CHAVE = dict(zip(sorted(codigos), regimes))
    with open(CHAVE_ARQ, 'w') as f:
        json.dump(CHAVE, f, indent=2)
    print('Três códigos de regime sorteados. A chave foi gravada em resultados/semana6_chave e não será mostrada.')
CODIGOS = sorted(CHAVE)
print('Códigos:', ' '.join(CODIGOS))

## Inferência

Uma execução por vez, em ordem sorteada. Para cada uma, grava `run_<código>_s<semente>_<conjunto>.npz` com `chaves`, `tamanhos` e `margem` (float32). Nada dos alvos é impresso além de contagens.

In [ ]:
if 'CONJ' not in globals() or 'ROTULOS' not in globals():
    raise RuntimeError('Faltam os conjuntos na memória. Rode a célula 6 (leitura dos conjuntos) antes desta, depois da 7 e da 8.')

import psutil
LIMITE_RAM_PCT = 62
DIFS_VAL = []

def limpar(msg):
    return re.sub(r'(?<![A-Za-z])R[123](?!\d)', 'Rx', str(msg))

def arquivo_run(cod, sem, conj):
    return f'{SAIDA}/run_{cod}_s{sem}_{conj}.npz'

def executar(cod, sem):
    faltam = [c for c in CONJ if not os.path.exists(arquivo_run(cod, sem, c))]
    if not faltam:
        print(f'{cod} s{sem}: já feita.')
        return
    modelo = carregar_modelo(CHAVE[cod], sem)
    for conj in faltam:
        with em_silencio():
            margem = inferir(modelo, CONJ[conj])
        r = ROTULOS[conj]
        assert margem.dtype == np.float32 and margem.size == int(r['tamanhos'].sum()), f'{conj}: tamanho da margem'
        assert np.isfinite(margem).all(), f'{conj}: margens não finitas, não grave'
        if conj == 'val':
            dif_max, trocas = comparar_val(CHAVE[cod], sem, r['chaves'], r['tamanhos'], margem)
            assert dif_max <= LIMITE_DIF_VAL, f'val: diferença de {dif_max:.3f} contra a inferência do treino'
            DIFS_VAL.append((dif_max, trocas))
            print(f'{cod} s{sem}: val conferida com a do treino.')
        gravar(arquivo_run(cod, sem, conj), chaves=r['chaves'], tamanhos=r['tamanhos'], margem=margem)
    print(f'{cod} s{sem}: {len(faltam)} conjuntos gravados (RAM em {psutil.virtual_memory().percent:.0f}%).')
    del modelo
    gc.collect(); torch.cuda.empty_cache()

ORDEM = [(c, s) for c in CODIGOS for s in SEMENTES]
secrets.SystemRandom().shuffle(ORDEM)
for cod, sem in ORDEM:
    pct = psutil.virtual_memory().percent
    if pct > LIMITE_RAM_PCT:
        print(f'PAREI: RAM em {pct:.0f}%. Reinicie a sessão e rode da célula 2 em diante; o que está feito é pulado.')
        break
    try:
        executar(cod, sem)
    except Exception as erro:
        raise RuntimeError(f'{cod} s{sem}: {type(erro).__name__}: {limpar(erro)}') from None
if DIFS_VAL:
    print(f'\nConferência da val em {len(DIFS_VAL)} execuções: diferença máxima {max(d for d, _ in DIFS_VAL):.1e}, '
          f'{sum(t for _, t in DIFS_VAL)} pixels com sinal trocado no total.')
print('\nGuarde a saída desta célula.')

## Aumento em tempo de teste (opcional)

Linha de base secundária do pré-registro: as 8 transformações diedrais, com a margem igual ao logit da probabilidade média de água. Leva cerca de 8 vezes o tempo da inferência simples. Para rodar, mude `FAZER_TTA` para `True`. Sem ela, rode a análise com `--sem-tta`.

In [ ]:
# OPCIONAL. Aumento em tempo de teste com as 8 transformações diedrais (linha de base secundária).
# Leva cerca de 8 vezes o tempo da inferência simples. Sem esta célula, rode a análise com --sem-tta.
FAZER_TTA = False

if FAZER_TTA:
    assert shutil.disk_usage(MEUDRIVE)[2] > 12e9, 'Espaço livre no Drive insuficiente para o TTA.'
    for cod, sem in ORDEM:
        if psutil.virtual_memory().percent > LIMITE_RAM_PCT:
            print('PAREI: RAM alta. Reinicie a sessão, rode da célula 2 até a 8 e depois esta célula de novo.')
            break
        if all('margem_tta' in np.load(arquivo_run(cod, sem, c), allow_pickle=False).files for c in CONJ):
            print(f'{cod} s{sem}: TTA já feito.')
            continue
        try:
            modelo = carregar_modelo(CHAVE[cod], sem)
            for conj in CONJ:
                caminho = arquivo_run(cod, sem, conj)
                z = dict(np.load(caminho, allow_pickle=False))
                if 'margem_tta' in z:
                    continue
                with em_silencio():
                    m_tta = inferir(modelo, CONJ[conj], tta=True)
                assert m_tta.shape == z['margem'].shape and np.isfinite(m_tta).all()
                z['margem_tta'] = m_tta
                gravar(caminho, **z)
            del modelo
            gc.collect(); torch.cuda.empty_cache()
        except Exception as erro:
            raise RuntimeError(f'{cod} s{sem}: {type(erro).__name__}: {limpar(erro)}') from None
        print(f'{cod} s{sem}: TTA gravado.')
else:
    print('TTA desligado. Mude FAZER_TTA para True se quiser rodar.')

## Conferência final

Confere os 60 arquivos (15 execuções × 4 conjuntos): unidades e tamanhos iguais aos dos rótulos, margens finitas em float32 e nenhuma margem repetida entre execuções. Grava o manifesto de hashes e o `info_inferencia.json`. **Rode de novo depois do TTA**, se ele foi feito.

In [ ]:
if 'CONJ' not in globals() or 'ROTULOS' not in globals():
    raise RuntimeError('Faltam os conjuntos na memória. Rode a célula 6 (leitura dos conjuntos) antes desta, depois da 7 e da 8.')
manifesto, hashes_val = [], {}
tta_feito = True
for cod in CODIGOS:
    for sem in SEMENTES:
        for conj in CONJ:
            caminho = arquivo_run(cod, sem, conj)
            assert os.path.exists(caminho), f'Falta {os.path.basename(caminho)}. Rode a célula anterior de novo.'
            z = np.load(caminho, allow_pickle=False)
            r = ROTULOS[conj]
            assert [str(c) for c in z['chaves']] == [str(c) for c in r['chaves']] and np.array_equal(z['tamanhos'], r['tamanhos'])
            m = z['margem']
            tta_feito &= 'margem_tta' in z.files
            if 'margem_tta' in z.files:
                assert z['margem_tta'].dtype == np.float32 and z['margem_tta'].shape == m.shape and np.isfinite(z['margem_tta']).all()
            assert m.dtype == np.float32 and m.size == int(r['tamanhos'].sum()) and np.isfinite(m).all()
            if conj == 'val':
                hashes_val[(cod, sem)] = hashlib.sha256(m.tobytes()).hexdigest()
            manifesto.append((os.path.basename(caminho), os.path.getsize(caminho), sha256_arquivo(caminho)))
assert len(set(hashes_val.values())) == len(hashes_val), 'Duas execuções com a mesma margem na val.'
for conj in CONJ:
    arq = f'{SAIDA}/labels_{conj}.npz'
    manifesto.append((os.path.basename(arq), os.path.getsize(arq), sha256_arquivo(arq)))
with open(f'{SAIDA}/manifesto_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['arquivo', 'bytes', 'sha256'])
    w.writerows(manifesto)
n_tta = sum('margem_tta' in np.load(arquivo_run(c, s, k), allow_pickle=False).files for c in CODIGOS for s in SEMENTES for k in CONJ)
if 0 < n_tta < len(CODIGOS) * len(SEMENTES) * len(CONJ):
    print(f'ATENÇÃO: TTA em {n_tta} de {len(CODIGOS) * len(SEMENTES) * len(CONJ)} arquivos. Termine a célula 10 ou rode a análise com --sem-tta.')
INFO = {'gerado_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'preregistro_commit': '26cc8b7', 'codigos': CODIGOS, 'sementes': list(SEMENTES),
        'conjuntos': {c: len(ROTULOS[c]['chaves']) for c in ROTULOS},
        'listas_sen1floods11': {c: v[2] for c, v in LISTAS.items()},
        'tabelas_janelas': {'t2_janelas.csv': '3c21970835f2d43cd436be9616c451ebbff5fcea138b7401d3867f0901643b2c',
                            't2_janelas_nuvem050.csv': '3b46e703738042b235b68a12600cfc0c7bf917b3cf5dec7efd8b924a02e132a6'},
        'backbone_sha256': SHA_BACKBONE, 'borda': 'a ate 2 pixels (janela 5 x 5) de pixel da outra classe, rotulo -1 nao forma borda',
        'precisao_inferencia': 'float32', 'tta': tta_feito,
        'versoes': {q: version(q) for q in ('terratorch', 'torchgeo', 'peft', 'torch', 'numpy')}}
json.dump(INFO, open(f'{SAIDA}/info_inferencia.json', 'w'), indent=2, ensure_ascii=False)
print(f'{len(CODIGOS) * len(SEMENTES)} execuções x {len(CONJ)} conjuntos conferidos: tamanhos, finitude e margens distintas.')
print('Manifesto e info_inferencia.json gravados em resultados/semana6/margens.')
print('Não abra a pasta resultados/semana6_chave antes da análise.')